# Fuentes con memoria y procesos estocasticos

Exploramos fuentes de Markov de orden k, tasa de entropia y comparacion con fuentes i.i.d.

Articulo: `02/14-procesos-estocasticos-y-fuentes-con-memoria.md`

In [ ]:
import math
import random
import numpy as np
from collections import defaultdict, Counter

## 1. Fuente i.i.d. vs fuente de Markov de orden 1

Una fuente i.i.d. produce simbolos independientes. Una fuente de Markov de orden 1 genera cada simbolo condicionado al anterior.

In [ ]:
# Fuente i.i.d. con distribucion sobre {A,B,C}
def fuente_iid(p, n):
    simbolos = list(p.keys())
    probs = list(p.values())
    return random.choices(simbolos, weights=probs, k=n)

# Fuente de Markov de orden 1
def fuente_markov1(matriz_trans, estado_inicial, n):
    secuencia = [estado_inicial]
    estado = estado_inicial
    for _ in range(n-1):
        siguiente = random.choices(list(matriz_trans[estado].keys()),
                                   weights=list(matriz_trans[estado].values()))[0]
        secuencia.append(siguiente)
        estado = siguiente
    return secuencia

# Ejemplo: bigramas del espanol simplificado
p_iid = {'a': 0.15, 'e': 0.13, 'o': 0.09, 'n': 0.07, 's': 0.07,
         'r': 0.07, 'i': 0.06, 'l': 0.05, 't': 0.05, ' ': 0.17,
         'otro': 0.10}

trans_markov = {
    'a': {'n': 0.2, 's': 0.15, 'r': 0.15, 'l': 0.1, ' ': 0.4},
    'e': {'n': 0.2, 's': 0.15, 'r': 0.15, 'l': 0.1, ' ': 0.4},
    'n': {'a': 0.25, 'e': 0.25, 'o': 0.15, ' ': 0.35},
    's': {'e': 0.2, ' ': 0.5, 'a': 0.2, 'o': 0.1},
    ' ': {'l': 0.15, 'e': 0.15, 'a': 0.15, 'n': 0.1, 's': 0.1, 'r': 0.1, 'otro': 0.25},
    'r': {'a': 0.2, 'e': 0.2, ' ': 0.3, 'o': 0.15, 'i': 0.15},
    'l': {'a': 0.25, 'e': 0.25, 'o': 0.15, ' ': 0.35},
    'otro': {' ': 0.4, 'a': 0.2, 'e': 0.2, 'o': 0.2},
    'i': {'n': 0.2, 's': 0.1, 'o': 0.1, ' ': 0.4, 'otro': 0.2},
    'o': {'n': 0.2, 's': 0.1, ' ': 0.4, 'r': 0.15, 'otro': 0.15},
    't': {'e': 0.25, 'a': 0.2, ' ': 0.3, 'r': 0.15, 'i': 0.1},
}

random.seed(42)
seq_iid = fuente_iid(p_iid, 500)
seq_markov = fuente_markov1(trans_markov, ' ', 500)

print("Primeros 100 simbolos (i.i.d.):")
print(''.join(seq_iid[:100]))
print("\nPrimeros 100 simbolos (Markov orden 1):")
print(''.join(seq_markov[:100]))

## 2. Tasa de entropia de la fuente

Para una fuente de Markov ergodica con distribucion estacionaria pi y matriz de transicion P:

$$\bar{H} = -\sum_i \pi_i \sum_j P_{ij} \log_2 P_{ij}$$

In [ ]:
def distribucion_estacionaria(trans, estados, max_iter=1000, tol=1e-10):
    """Distribucion estacionaria por iteracion de potencia."""
    pi = {e: 1/len(estados) for e in estados}
    for _ in range(max_iter):
        nueva_pi = defaultdict(float)
        for e in estados:
            for succ, p in trans.get(e, {}).items():
                nueva_pi[succ] += pi[e] * p
        diff = sum(abs(nueva_pi.get(e,0) - pi[e]) for e in estados)
        pi = dict(nueva_pi)
        if diff < tol:
            break
    total = sum(pi.values())
    return {e: v/total for e, v in pi.items()}

def tasa_entropia_markov(trans, pi):
    """H_bar = -sum_i pi_i * sum_j P_ij log P_ij"""
    H_bar = 0
    for estado, p_estado in pi.items():
        for succ, p_trans in trans.get(estado, {}).items():
            if p_trans > 0:
                H_bar -= p_estado * p_trans * math.log2(p_trans)
    return H_bar

estados = list(trans_markov.keys())
pi = distribucion_estacionaria(trans_markov, estados)

print("Distribucion estacionaria pi:")
for e, p in sorted(pi.items(), key=lambda x: -x[1]):
    print(f"  '{e}': {p:.4f}")

H_bar = tasa_entropia_markov(trans_markov, pi)
H_iid = -sum(p * math.log2(p) for p in p_iid.values() if p > 0)
print(f"\nTasa de entropia (Markov): H_bar = {H_bar:.4f} bits/simbolo")
print(f"Entropia fuente i.i.d.:              H    = {H_iid:.4f} bits/simbolo")
print(f"Reduccion por memoria: {(H_iid - H_bar)/H_iid:.2%}")

## 3. Estimacion de la tasa de entropia por k-gramas

Podemos estimar H_bar empiricamente usando la entropia condicional de k-gramas:
$$\hat{H}_k = H(X_k | X_{k-1}, \ldots, X_1) \approx \bar{H} \text{ para } k \text{ grande}$$

In [ ]:
def estimar_tasa_entropia_kgrama(secuencia, k):
    """Estima la tasa de entropia usando entropia condicional de k-gramas."""
    kgramas = Counter(tuple(secuencia[i:i+k]) for i in range(len(secuencia)-k+1))
    k1gramas = Counter(tuple(secuencia[i:i+k-1]) for i in range(len(secuencia)-k+2))
    
    H_cond = 0
    total = sum(kgramas.values())
    for kgrama, cnt in kgramas.items():
        contexto = kgrama[:-1]
        p_contexto = k1gramas[contexto] / (len(secuencia) - k + 2)
        p_cond = cnt / k1gramas[contexto] if k1gramas[contexto] > 0 else 0
        if p_cond > 0:
            H_cond -= p_contexto * p_cond * math.log2(p_cond)
    return H_cond

# Secuencia larga para mejor estimacion
random.seed(123)
seq_larga = fuente_markov1(trans_markov, ' ', 10000)

print("Estimacion de la tasa de entropia por k-gramas:")
print(f"  Tasa real H_bar = {H_bar:.4f} bits/simbolo")
for k in range(1, 6):
    estimado = estimar_tasa_entropia_kgrama(seq_larga, k)
    print(f"  k={k}: H_hat = {estimado:.4f}")

## 4. Fuentes de Markov de orden k

Generalizamos a fuentes de orden k: cada simbolo depende de los k anteriores.
A mayor orden, la tasa de entropia converge a la del proceso real.

In [ ]:
def fuente_markov_orden_k(corpus, k, n):
    """Aprende un modelo de Markov de orden k de un corpus y genera n simbolos."""
    # Aprender transiciones
    trans = defaultdict(Counter)
    for i in range(len(corpus) - k):
        contexto = tuple(corpus[i:i+k])
        siguiente = corpus[i+k]
        trans[contexto][siguiente] += 1
    
    # Generar
    inicio = random.randint(0, len(corpus)-k-1)
    resultado = list(corpus[inicio:inicio+k])
    for _ in range(n - k):
        contexto = tuple(resultado[-k:])
        if contexto not in trans:
            break
        siguientes = list(trans[contexto].keys())
        pesos = list(trans[contexto].values())
        resultado.append(random.choices(siguientes, weights=pesos)[0])
    return resultado

# Corpus de ejemplo: secuencia larga de Markov como "corpus"
corpus = seq_larga

print("Generacion con modelos de orden k (muestra de 80 simbolos):")
random.seed(99)
for k in [1, 2, 3, 4]:
    generado = fuente_markov_orden_k(corpus, k, 80)
    print(f"  Orden {k}: {''.join(generado)}")

print()
print("A mayor orden, la generacion captura mejor las dependencias locales.")
print("La tasa de entropia decrece con el orden hasta converger a H_bar del proceso real.")